# AIT303 Assignment 1 — Aspect-Based Sentiment Analysis

**Author:** Oh Kai Wen

This notebook implements the data preparation and preprocessing pipeline for the IMDB 50K movie reviews dataset. It covers dataset loading, text cleaning, stemming, and lemmatization.

## 1. Setup and Imports

In [1]:
# Run: pip install nltk pandas numpy jupyter

import ssl
import os
import shutil

# SSL context workaround for NLTK downloads (prevents SSLError in some environments)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download NLTK resources (suppress verbose output with quiet=True)
import nltk
nltk.download('punkt', quiet=True)                       # Tokenizer for word/sentence splitting
nltk.download('stopwords', quiet=True)                   # Common English stopwords list (~179 words)
nltk.download('wordnet', quiet=True)                     # WordNet lexical database for lemmatization
nltk.download('averaged_perceptron_tagger', quiet=True)  # POS tagger for part-of-speech-aware lemmatization

import pandas as pd           # Data manipulation and analysis (DataFrame)
import numpy as np            # Numerical computing and array operations
import re                     # Regular expressions for text cleaning and pattern matching
from nltk.tokenize import word_tokenize          # Word tokenizer: splits text into tokens
from nltk.corpus import stopwords                # Stopwords: common words to filter out
from nltk.stem import PorterStemmer              # Porter Stemmer: algorithmic suffix stripping
from nltk.stem import WordNetLemmatizer          # WordNet Lemmatizer: dictionary-based root form
from nltk import pos_tag                          # POS tagger: identifies noun/verb/adjective etc.
from nltk.corpus import wordnet                   # WordNet constants for POS tag mapping

print("Setup complete. NLTK resources downloaded.")

Setup complete. NLTK resources downloaded.


In [2]:
# Create data_asg/ directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Check if IMDB Dataset.csv is in project root and move it to data_asg/ directory
if os.path.exists('IMDB Dataset.csv'):
    shutil.move('IMDB Dataset.csv', 'data_asg/IMDB Dataset.csv')
    print("Moved IMDB Dataset.csv to data_asg/ directory.")
elif os.path.exists('data_asg/IMDB Dataset.csv'):
    print("IMDB Dataset.csv already in data_asg/ directory.")
else:
    print("WARNING: IMDB Dataset.csv not found. Please download from Kaggle and place in data_asg/ directory.")

IMDB Dataset.csv already in data_asg/ directory.


## 2. Load and Inspect Dataset

In [3]:
# Load IMDB 50K movie reviews dataset into a DataFrame
df = pd.read_csv('data_asg/IMDB Dataset.csv')

# Display the shape (rows, columns) — expect (50000, 2)
print(f"Shape: {df.shape}")

# Show column names — expect ['review', 'sentiment']
print(f"\nColumns: {df.columns.tolist()}")

# Check for missing values — expect 0 for both columns
print(f"\nMissing values:\n{df.isnull().sum()}")

# Display class balance — expect 25000 positive, 25000 negative
print(f"\nClass balance:\n{df['sentiment'].value_counts()}")

Shape: (50000, 2)

Columns: ['review', 'sentiment']

Missing values:
review       0
sentiment    0
dtype: int64

Class balance:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
# Check unique sentiment values (should be 'positive' and 'negative' only)
print(f"Sentiment values: {df['sentiment'].unique()}")

Sentiment values: <StringArray>
['positive', 'negative']
Length: 2, dtype: str


In [5]:
# Display first 5 rows to inspect sample reviews
print("First 5 reviews:")
df.head()

First 5 reviews:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
# Display last 5 rows to inspect tail-end reviews
print("Last 5 reviews:")
df.tail()

Last 5 reviews:


,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


### Dataset Summary

The IMDB 50K dataset is loaded successfully. Key properties:
- **Shape:** 50,000 rows × 2 columns (`review`, `sentiment`)
- **Missing values:** 0 — the dataset is complete with no null entries
- **Class balance:** Perfectly balanced — 25,000 positive and 25,000 negative reviews
- **Sentiment labels:** Binary — `positive` and `negative` only

The data is clean, balanced, and ready for preprocessing.

## 3. Text Cleaning Pipeline

### 3.1 Define Cleaning Function

In [7]:
# Define text cleaning function: lowercase, remove HTML, remove non-alpha, normalize whitespace
def clean_text(text):
    """Clean raw review text: lowercase, remove HTML tags, remove non-alpha characters, normalize whitespace."""
    text = text.lower()                                     # Convert to lowercase
    text = re.sub(r'<.*?>', '', text)                       # Remove HTML tags like <br />
    text = re.sub(r'[^a-zA-Z]', ' ', text)                 # Remove numbers, punctuation, special characters
    text = re.sub(r'\s+', ' ', text).strip()              # Normalize multiple spaces to single space
    return text

### 3.2 Before/After Sample

In [8]:
# Show before/after cleaning for a sample review
sample_idx = 0
print(f"Original ({len(df['review'].iloc[sample_idx])} chars):")
print(df['review'].iloc[sample_idx][:200])
print(f"\nCleaned ({len(clean_text(df['review'].iloc[sample_idx]))} chars):")
print(clean_text(df['review'].iloc[sample_idx])[:200])

Original (1761 chars):
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

Cleaned (1676 chars):
one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its br


### 3.3 Apply to Entire Dataset

In [9]:
# Apply the cleaning function to all 50K reviews (this may take a moment)
df['cleaned'] = df['review'].apply(clean_text)

# Confirm the new column was added
print(f"Cleaning complete. Column 'cleaned' added to DataFrame.")
print(f"Shape: {df.shape}")
print(f"Processed {len(df)} reviews.")

Cleaning complete. Column 'cleaned' added to DataFrame.
Shape: (50000, 3)
Processed 50000 reviews.


### 3.4 Verify Cleaning Results

In [10]:
# Verify no HTML tags remain in cleaned text
html_tag_count = df['cleaned'].str.contains('<br').sum()
print(f"HTML tag removal verified: {html_tag_count} <br> tags found in cleaned column (expected: 0)")

# Display side-by-side comparison of original vs cleaned for first 3 reviews
df[['review', 'cleaned']].head(3)

HTML tag removal verified: 0 <br> tags found in cleaned column (expected: 0)


,review,cleaned
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...


## 4. Stemming Output

### 4.1 Define Stemming Function

In [11]:
# Initialize Porter Stemmer (algorithmic suffix stripping — per D-04)
stemmer = PorterStemmer()

# Load NLTK English stopwords (~179 words — per D-07)
stop_words = set(stopwords.words('english'))

# Define stemming function: tokenize -> remove stopwords -> apply Porter stemming
def stem_text(text):
    """Tokenize, remove stopwords, apply Porter stemming."""
    tokens = word_tokenize(text)                          # Tokenize into individual words
    tokens = [t for t in tokens if t not in stop_words]  # Remove stopwords ('the', 'and', 'is', etc.)
    stems = [stemmer.stem(t) for t in tokens]            # Apply Porter stemming algorithm
    return ' '.join(stems)                                # Return space-joined string

### 4.2 Sample Comparison (Before/After)

In [12]:
# Compare cleaned vs stemmed text for a sample review
sample = df['cleaned'].iloc[0]
print(f"Cleaned ({len(sample)} chars):")
print(sample[:200])
print(f"\nStemmed ({len(stem_text(sample))} chars):")
print(stem_text(sample)[:200])

Cleaned (1676 chars):
one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its br

Stemmed (990 chars):
one review mention watch oz episod hook right exactli happen first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid show pull punch regard drug sex violenc 


### 4.3 Apply to Dataset

In [13]:
# Apply stemming to all cleaned reviews (this may take a moment)
df['stemmed'] = df['cleaned'].apply(stem_text)

# Confirm the new column
print(f"Stemming complete. Column 'stemmed' added.")
print(f"Shape: {df.shape}")

Stemming complete. Column 'stemmed' added.
Shape: (50000, 4)


### 4.4 Verify Stemming Effects

In [14]:
# Display side-by-side comparison: review -> cleaned -> stemmed for first 2 reviews
df[['review', 'cleaned', 'stemmed']].head(2)

,review,cleaned,stemmed
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...,wonder littl product film techniqu unassum old...


## 5. Lemmatization Output

### 5.1 Define POS Tag Mapper

In [15]:
# Map NLTK Penn Treebank POS tag to WordNet POS tag for accurate lemmatization (per D-03)
def get_wordnet_pos(treebank_tag):
    """Map NLTK Penn Treebank POS tag to WordNet POS tag constant."""
    if treebank_tag.startswith('J'):     # Adjective (JJ, JJR, JJS)
        return wordnet.ADJ
    if treebank_tag.startswith('V'):     # Verb (VB, VBD, VBG, VBN, VBP, VBZ)
        return wordnet.VERB
    if treebank_tag.startswith('N'):     # Noun (NN, NNS, NNP, NNPS)
        return wordnet.NOUN
    if treebank_tag.startswith('R'):     # Adverb (RB, RBR, RBS)
        return wordnet.ADV
    return wordnet.NOUN                  # Default: noun

### 5.2 Define Lemmatization Function

In [16]:
# Initialize WordNet lemmatizer (per D-02 — NLTK, not spaCy)
lemmatizer = WordNetLemmatizer()

# Load NLTK stopwords (per D-07)
stop_words = set(stopwords.words('english'))

# Define lemmatization function: tokenize -> stopword removal -> POS tag -> lemmatize with POS
def lemmatize_text(text):
    """Tokenize, remove stopwords, POS tag, apply WordNet lemmatization with POS."""
    tokens = word_tokenize(text)                           # Tokenize into individual words
    tokens = [t for t in tokens if t not in stop_words]   # Remove stopwords ('the', 'and', 'is', etc.)
    pos_tags = pos_tag(tokens)                             # Get POS tags (per D-03: noun/verb/adjective detection)
    lemmas = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    return ' '.join(lemmas)                                 # Return space-joined string

### 5.3 Sample Comparison (Before/After)

In [17]:
# Compare all 4 stages: cleaned -> stemmed -> lemmatized for a sample review
sample = df['cleaned'].iloc[0]
print(f"Cleaned ({len(sample)} chars):")
print(sample[:200])
print(f"\nStemmed ({len(stem_text(sample))} chars):")
print(stem_text(sample)[:200])
print(f"\nLemmatized ({len(lemmatize_text(sample))} chars):")
print(lemmatize_text(sample)[:200])

Cleaned (1676 chars):
one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its br

Stemmed (990 chars):
one review mention watch oz episod hook right exactli happen first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid show pull punch regard drug sex violenc 

Lemmatized (1046 chars):
one reviewer mention watch oz episode hook right exactly happen first thing strike oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug 


### 5.4 Apply to Dataset

In [18]:
# Apply lemmatization to all cleaned reviews (this may take a moment)
df['lemmatized'] = df['cleaned'].apply(lemmatize_text)

# Confirm the new column
print(f"Lemmatization complete. Column 'lemmatized' added.")
print(f"Shape: {df.shape}")

Lemmatization complete. Column 'lemmatized' added.
Shape: (50000, 5)


### 5.5 Final Data Summary

In [19]:
# Final preprocessing summary
print("Preprocessing complete!")
print(f"Final DataFrame shape: {df.shape}")                          # Expected: (50000, 5)
print(f"Columns: {df.columns.tolist()}")                           # Expected: [review, sentiment, cleaned, stemmed, lemmatized]
print("\nFirst 3 reviews — all 4 text stages:")
df.head(3)

Preprocessing complete!
Final DataFrame shape: (50000, 5)
Columns: ['review', 'sentiment', 'cleaned', 'stemmed', 'lemmatized']

First 3 reviews — all 4 text stages:


,review,sentiment,cleaned,stemmed,lemmatized
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...,one reviewer mention watch oz episode hook rig...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production the filming tech...,wonder littl product film techniqu unassum old...,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...,think wonderful way spend time hot summer week...


In [20]:
# Compare token/word counts across preprocessing stages for first 3 reviews
print("Token count comparison (word count in each stage):")
for i in range(3):
    raw_words = len(df['review'].iloc[i].split())
    cleaned_words = len(df['cleaned'].iloc[i].split())
    stemmed_words = len(df['stemmed'].iloc[i].split())
    lemmatized_words = len(df['lemmatized'].iloc[i].split())
    print(f"  Review {i}: raw={raw_words} -> cleaned={cleaned_words} -> stemmed={stemmed_words} -> lemmatized={lemmatized_words}")

Token count comparison (word count in each stage):
  Review 0: raw=307 -> cleaned=313 -> stemmed=162 -> lemmatized=162
  Review 1: raw=162 -> cleaned=160 -> stemmed=86 -> lemmatized=86
  Review 2: raw=166 -> cleaned=167 -> stemmed=84 -> lemmatized=84
